# Hyperparameter search with Optuna

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TXSTCODEPLAYGROUND/CodeCS4337Fall2026/blob/main/LitWBHSTrainingBasicNeuralNetwork/hyperparameter_search.ipynb)

This notebook searches the hyperparameters of the fully connected network of `LitWBHSTrainingBasicNeuralNetwork` with [Optuna](https://optuna.org), then trains the best settings found. Every trial is logged to Weights & Biases (W&B).

Run the cells from top to bottom:

1. **Set up** the runtime (Colab only): GPU, Google Drive, the repo, and your W&B key.
2. **Look at the search space**: what is searched, and in which ranges.
3. **Run the search**: Optuna trains many settings and keeps the best one.
4. **Read the results**: the best trial, Optuna's plots, and the W&B dashboard.
5. **Train the best config** and test it.

**Running locally instead of Colab:** skip section 1, start Jupyter from the repo's `.venv` (see the main README), and open this notebook from the project folder.

## 1. Setup (Colab)

The same setup as in the starter notebook. First choose a GPU: *Runtime → Change runtime type → GPU*. Then run these cells once per session (and again after the runtime resets): mount Drive, set up the repo, change directory, and load API keys.

Results, including the Optuna study, go to Google Drive, so a search interrupted by a disconnect can be continued (see the tips at the end).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%%bash
set -e

cd /content
if [ -d CodeCS4337Fall2026/.git ]; then
  git -C CodeCS4337Fall2026 pull --ff-only
else
  git clone https://github.com/TXSTCODEPLAYGROUND/CodeCS4337Fall2026.git
fi
cd CodeCS4337Fall2026

# Colab settings: data on the local disk, results in Google Drive.
cp .envcolab .env

# Install only packages Colab does not already have (versions are not pinned).
grep -vE '^\s*(#|$)' requirements.txt \
  | sed -E 's/[<>=!~;[ ].*//' \
  | while read -r pkg; do
      if pip show "$pkg" > /dev/null 2>&1; then
        echo "skip $pkg (already installed)"
      else
        pip install -q "$pkg" && echo "installed $pkg"
      fi
    done

In [ ]:
# The bash cell above cannot change the notebook's working directory, so do it here.
%cd /content/CodeCS4337Fall2026

### Load your W&B key (optional)

Add your key as a Colab **Secret** named `WANDB_API_KEY` (key icon in the left sidebar → *Add new secret*). Never type it into a cell. The cell below loads `.env`, then looks up missing keys in your Secrets, adds them to this notebook's environment, and saves them to `.env` so terminal runs find them too. Without a key, every trial is logged to W&B offline and can be uploaded later with `wandb sync`.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from google.colab import userdata

ENV_FILE = Path("/content/CodeCS4337Fall2026/.env")
KEYS = ["WANDB_API_KEY"]

if load_dotenv(ENV_FILE):
    print(f"Loaded settings from {ENV_FILE}")
else:
    print(f"No settings in {ENV_FILE}: run the setup cells first")

for key in KEYS:
    if os.getenv(key):
        print(f"{key}: found in .env")
        continue
    try:
        os.environ[key] = userdata.get(key)
    except Exception:  # noqa: BLE001 (Secret not added or access not granted)
        print(f"{key}: not found, W&B will log offline")
        continue
    with ENV_FILE.open("a") as env_file:
        env_file.write(f"\n{key}={os.environ[key]}\n")
    print(f"{key}: loaded from Colab Secrets and saved to .env")

## 2. The search space

This cell makes the project importable (in Colab and locally) and imports its two entry points: `search` runs the Optuna study, `main` trains and tests one config.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = Path("/content/CodeCS4337Fall2026") if IN_COLAB else Path.cwd().parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from LitWBHSTrainingBasicNeuralNetwork import main
from LitWBHSTrainingBasicNeuralNetwork.search import search
from LitWBHSTrainingBasicNeuralNetwork.utils import CONFIGS_DIR

print((CONFIGS_DIR / "search01.json").read_text())

The search config has three parts:

- `base_config`: the training config every trial starts from. Everything not searched (seed, validation split, W&B project) comes from it.
- `study`: how many trials to run (`n_trials`), an optional time limit (`timeout_minutes`), the sampler's seed, the pruner settings, and whether trials are logged to W&B.
- `search_space`: a range `[low, high]` or a list of choices for every hyperparameter.

| Hyperparameter | What it controls |
| --- | --- |
| `n_layers`, `units` | Number of hidden layers, and the units of each one (picked per layer) |
| `activation` | ReLU, Leaky ReLU, GELU, or tanh after every hidden layer |
| `dropout` | Dropout probability after every hidden layer |
| `batch_size` | Samples per training step |
| `epochs` | Maximum number of epochs |
| `optimizer`, `lr`, `momentum` | Adam, AdamW, SGD, or RMSprop; the learning rate (searched on a log scale); momentum (SGD and RMSprop only) |
| `regularization`, `l1`, `l2` | No penalty, L1 (pushes weights to zero), L2 (weight decay, shrinks weights), or both, with their weights (log scale) |
| `scheduler`, `step_size`, `gamma` | Constant learning rate, step decay, cosine decay, or reduce-on-plateau, with their settings |
| `early_stopping`, `patience` | Whether to stop when validation accuracy stops improving, and after how many epochs |

Some hyperparameters only exist for some choices, e.g. `momentum` only for SGD and RMSprop. Optuna handles this naturally: the search space is defined by running Python code (`suggest_config` in `search.py`), so an `if` decides what is sampled.

**To change the search**, copy `search01.json` in the file browser (left sidebar, `CodeCS4337Fall2026/LitWBHSTrainingBasicNeuralNetwork/configs/`), e.g. to `search02.json`, edit it, and pass its name below. To fix a hyperparameter instead of searching it, give a single choice (`["adam"]`) or a range with equal ends (`[10, 10]`). Fewer searched hyperparameters means fewer trials are needed.

## 3. Run the search

Each trial trains one setting on the training split and is scored by its best **validation** accuracy. The test set is never used during the search: choosing settings by their test score would make that score optimistic.

- Optuna's **TPE sampler** picks each trial's settings from how earlier trials did: after the first random trials, it samples more where the scores were good.
- The **median pruner** stops a trial early when its validation accuracy after an epoch is below the median of earlier trials at the same epoch. Pruned trials save time for promising ones.

A trial takes from a few seconds to about a minute on a GPU, so 30 trials take roughly 10 to 20 minutes. Try `N_TRIALS = 3` first to check that everything works. Running the cell again **adds** trials to the same study instead of starting over.

In [ ]:
SEARCH_CONFIG = "search01.json"
N_TRIALS = 30

study = search(SEARCH_CONFIG, n_trials=N_TRIALS)

**Or from a terminal** (the same search; the notebook stays free while it runs in Colab's terminal):

```bash
cd /content/CodeCS4337Fall2026
python -m LitWBHSTrainingBasicNeuralNetwork.search --config search01.json --n-trials 30
```

Then load the study in the notebook with the next cell (`N_TRIALS = 0` runs no new trial).

In [ ]:
# Reload a saved study (after a terminal search or a runtime reset) without new trials.
study = search(SEARCH_CONFIG, n_trials=0)

## 4. Read the results

The ten best trials, best first:

In [ ]:
from optuna.trial import TrialState

complete = study.get_trials(states=[TrialState.COMPLETE])
pruned = study.get_trials(states=[TrialState.PRUNED])
print(f"{len(complete)} complete, {len(pruned)} pruned\n")
for trial in sorted(complete, key=lambda t: t.value, reverse=True)[:10]:
    print(f"trial {trial.number:3d}  val_acc {trial.value:.4f}  {trial.params}\n")

### Optuna's plots

- **Optimization history:** each trial's score, and the best so far. It should rise quickly, then flatten: more trials then bring little.
- **Intermediate values:** the validation accuracy of every trial, epoch by epoch. Pruned trials are the short lines that stop early.
- **Hyperparameter importances:** how much of the variation in score each hyperparameter explains. Important ones deserve a finer search; unimportant ones can be fixed.
- **Slice plot:** the score against the value of one hyperparameter, one dot per trial. Where are the good learning rates?

In [ ]:
import warnings

import matplotlib.pyplot as plt
import optuna
from optuna.visualization import matplotlib as optuna_plots

# Optuna's matplotlib plots are marked experimental; the warning is not useful here.
warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)

optuna_plots.plot_optimization_history(study)
plt.show()
optuna_plots.plot_intermediate_values(study)
plt.show()
optuna_plots.plot_param_importances(study)
plt.show()
optuna_plots.plot_slice(study, params=["lr", "dropout", "optimizer", "n_layers"])
plt.show()

### The W&B dashboard

Every trial is a W&B run in the project `LitWBHSTrainingBasicNeuralNetwork`, grouped under the search's name (`search01`), with its settings in the run config and `best_val_acc` and `state` (`complete` or `pruned`) in the summary. On wandb.ai:

1. In the **Runs** table, group by `group`, show the columns you care about (`training.optimizer`, `training.lr`, `best_val_acc`, ...), and sort by `best_val_acc`.
2. Add a **Parallel coordinates** panel with the searched settings and `best_val_acc`: one line per trial, so you can follow the best lines back to their settings.
3. Add a **Parameter importance** panel for `best_val_acc`: W&B's own estimate of which settings matter.
4. The `val_acc` and `lr-...` charts overlay all trials, showing what the schedulers and early stopping did.

Without a key, upload the trials later with the `wandb sync` command printed during the search.

## 5. Train the best config

The search saved the best trial's settings as a normal training config, `configs/search01_best.json` (and a copy, `best_config.json`, next to the study in Drive). Its `found_by` entry records which trial it came from.

In [ ]:
best_config_name = f"{Path(SEARCH_CONFIG).stem}_best.json"
print((CONFIGS_DIR / best_config_name).read_text())

Train it like any other config: a full run with checkpoints, then **one** evaluation on the test set, which is the score to report. The run is logged to W&B in the group `search01_best`, next to the trials. The same from a terminal: `python -m LitWBHSTrainingBasicNeuralNetwork --config search01_best.json`.

In [ ]:
best_metrics = main(best_config_name)
print(f"Test accuracy of the best config: {best_metrics['test_acc']:.4f}")

**Compare with the starting point** (optional): `config01.json` holds the default settings (Adam, learning rate 0.001, two hidden layers of 256 and 128 units, 10 epochs). Did the search beat it on the test set?

In [ ]:
baseline_metrics = main("config01.json")
print(
    f"Baseline {baseline_metrics['test_acc']:.4f}, best config {best_metrics['test_acc']:.4f}"
)

## Tips

**Long searches in Colab**
- The study is saved in Google Drive (`MyDrive/CodeCS4337Fall2026/runs/LitWBHSTrainingBasicNeuralNetwork/search01/study.db`) after every trial. After a disconnect, re-run the setup cells and the search cell: the finished trials are kept and new ones are added.
- `configs/search01_best.json` is in `/content`, which is deleted when the runtime resets. Get it back with the reload cell (`n_trials=0`), which rewrites it from the study.
- Set `timeout_minutes` in the search config to stop starting new trials after a time limit.
- The tips at the end of the starter notebook (idle disconnects, GPU quota) apply here too.

**Searching well**
- Start wide, then narrow: after a first search, read the importances and slice plots, fix the unimportant hyperparameters, and search the important ones in a smaller range (`search02.json`).
- To start a search over, use a new search config name, or delete its folder in Drive.
- Fewer epochs per trial make the search faster, but favor settings that learn fast rather than settings that end best.